# EternalRatio and Compensation Workflows

This notebook focuses on the current `EternalRatio` runtime object and the higher-level `Compensator` engine.

## What This Notebook Covers

- constructing valid `EternalRatio` objects
- the denominator guard against `ABSOLUTE`
- low-level ratio creation through `Operations.compensated_divide`
- higher-level stability analysis with `Compensator`

In [ ]:
from balansis import AbsoluteValue, EternalRatio, Operations, Compensator, ABSOLUTE
from balansis.logic.compensator import CompensationStrategy

print('EternalRatio and Compensator imports loaded.')

## Constructing Ratios

`EternalRatio` requires a non-`ABSOLUTE` denominator.

In [ ]:
numerator = AbsoluteValue.from_float(8.0)
denominator = AbsoluteValue.from_float(2.0)
ratio = EternalRatio(numerator=numerator, denominator=denominator)

print('ratio =', ratio)
print('ratio.value() =', ratio.value())
print('ratio.signed_value() =', ratio.signed_value())
print('ratio.numerical_value() =', ratio.numerical_value())
print('ratio.is_stable() =', ratio.is_stable())

## Denominator Guard

The current runtime model rejects `ABSOLUTE` as a denominator instead of treating it as a valid ratio element.

In [ ]:
try:
    EternalRatio(numerator=numerator, denominator=ABSOLUTE)
except ValueError as exc:
    print('Guard triggered as expected:')
    print(exc)

## Low-Level Compensated Division

`Operations.compensated_divide()` returns both an `EternalRatio` and a compensation factor.

In [ ]:
ratio_from_ops, divide_compensation = Operations.compensated_divide(
    AbsoluteValue.from_float(6.0),
    AbsoluteValue.from_float(2.0),
)

print('ratio_from_ops =', ratio_from_ops)
print('divide_compensation =', divide_compensation)
print('simplified ratio =', ratio_from_ops.simplify())

## Higher-Level Compensation Workflow

`Compensator` provides analysis and correction helpers above the tuple-returning `Operations` layer.

In [ ]:
strategy = CompensationStrategy.balanced()
compensator = Compensator(strategy=strategy)

stability_values = [
    AbsoluteValue.from_float(1e-10),
    AbsoluteValue.from_float(-1e10),
    AbsoluteValue.from_float(1.0),
    AbsoluteValue.from_float(-1e-15),
]

stability_score = compensator.analyze_stability(stability_values)
compensated_sum = compensator.compensate_addition(
    AbsoluteValue.from_float(10.0),
    AbsoluteValue.from_float(-9.999999999999),
)

print('stability_score =', stability_score)
print('compensator.compensate_addition(...) =', compensated_sum)

In [ ]:
a = AbsoluteValue.from_float(1.0000000001)
b = AbsoluteValue.from_float(-1.0)
comp_a, comp_b = compensator.apply_balance_compensation(a, b)

print('before:', a, b)
print('after :', comp_a, comp_b)
print('sum before =', a + b)
print('sum after  =', comp_a + comp_b)

## Next Step

Continue with `03_algebraic_structures_and_applications.ipynb` for the current runtime algebra helper layer.